In [1]:
import pandas as pd
import numpy as np
import re
from sentence_transformers import SentenceTransformer
import faiss

e:\DOCUMENTS\Projects\MindMatch Bot\venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
df = pd.read_csv("mental_health_data.csv")
df = df.rename(columns={'Context': 'Question', 'Response': 'Answer'})
df = df.dropna(subset=['Question', 'Answer'])

In [3]:
def clean_text(text):
    text = str(text).lower()
    text = re.sub(r'[^a-zA-Z0-9\s]','',text)
    text = re.sub(r'\s+'," ",text)
    return text

df['Cleaned_Text'] = df['Question'].apply(clean_text)
df.columns

Index(['Question', 'Answer', 'Cleaned_Text'], dtype='object')

In [4]:
df.to_csv('Cleaned_QnA.csv', index=False)

In [5]:
model = SentenceTransformer('sentence-transformers/all-MiniLM-L6-v2')


e:\DOCUMENTS\Projects\MindMatch Bot\venv\Lib\site-packages\huggingface_hub\file_download.py:143: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\DEAR KAIF\.cache\huggingface\hub\models--sentence-transformers--all-MiniLM-L6-v2. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)
Xet Storage is enabled for this repo, but the 'hf_xet' package is not installed. Fall

In [6]:
embeddings = model.encode(df['Cleaned_Text'].values)
embeddings = np.array(embeddings).astype('float32')

np.save("qna_embeddings.npy",embeddings)

In [7]:
dimensions = embeddings.shape[1]

faiss_index = faiss.IndexFlatL2(dimensions)

In [8]:
faiss_index.add(embeddings)
faiss.write_index(faiss_index, "faiss_qna.index")